# Mean-Variance Optimization (MVO) Portfolio — Final Version

Based on the two Top-6 selection CSVs produced by OLS and Random Forest models.

**Logic:**

- Each month T: use the 6 selected stocks + their predicted returns as μ
- **Covariance estimation (both models)**: estimate Σ as a **diagonal matrix**
  of each stock's rolling 36-month return variance, with off-diagonal
  correlations set to zero. Both the OLS and RF candidate pools rotate across
  a large number of different stocks each month (OLS: ~230, RF: ~99), so any
  two stocks selected in a given month share almost no overlapping return
  history. A full covariance matrix cannot be reliably estimated under these
  conditions; the diagonal approach is therefore applied consistently across
  both models.
- **Expected return vector (both models)**: raw predicted returns are
  **rank-transformed** to a [0, 0.10] scale before being passed to the
  optimizer. Because both models produce very small cross-sectional spread in
  predicted returns within any given month, rank transformation preserves the
  ordinal signal while removing sensitivity to spread magnitude. The scale
  factor is arbitrary and does not affect the MVO solution — multiplying all
  elements of μ by a constant leaves the max-Sharpe weights unchanged.
- Solve max-Sharpe MVO with long-only constraints and a **unified per-stock
  weight cap of 1/(N/2) = 1/3 ≈ 33%** for both models. For N = 6 stocks,
  this allows the optimizer to overweight any single position by at most 2×
  relative to the equal-weight baseline while preserving diversification across
  all six holdings. This is more conservative than the 40% upper bound
  suggested in the methodology (§9.4).
- Realized portfolio return = w\* · y_next (next-month excess returns)
- Compare OLS+MVO vs RF+MVO vs Equal-Weight benchmark on the same selected
  stocks

In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.6f}'.format)

## 1. Load Data

In [ ]:
# ── Adjust paths to your local setup ──────────────────────────────────────────
top6_ols = pd.read_csv('ols_top6_by_month_post2022.csv')
top6_rf  = pd.read_csv('rf_top6_by_month_post2022.csv')
# ──────────────────────────────────────────────────────────────────────────────

top6_ols['MthCalDt'] = pd.to_datetime(top6_ols['MthCalDt'])
top6_rf['MthCalDt']  = pd.to_datetime(top6_rf['MthCalDt'])

print('OLS shape:', top6_ols.shape,
      '| months:', top6_ols['DateKey'].nunique(),
      '| unique stocks:', top6_ols['PERMNO'].nunique())
print('RF  shape:', top6_rf.shape,
      '| months:', top6_rf['DateKey'].nunique(),
      '| unique stocks:', top6_rf['PERMNO'].nunique())

# Verify month alignment
ols_months = set(top6_ols['DateKey'].unique())
rf_months  = set(top6_rf['DateKey'].unique())
assert ols_months == rf_months, 'Month mismatch between OLS and RF datasets'
print(f'Month alignment: OK — {len(ols_months)} months, {min(ols_months)} to {max(ols_months)}')

## 2. MVO Core Functions

In [ ]:
# Weight cap: 1/(N/2) where N=6 → 1/3
N          = 6
WEIGHT_CAP = 1 / (N / 2)   # = 1/3 ≈ 0.3333
COV_WINDOW = 36

print(f'Weight cap: 1/(N/2) = 1/({N}/2) = {WEIGHT_CAP:.4f}')


def max_sharpe_weights(mu, cov, weight_cap):
    """
    Solve the maximum Sharpe Ratio portfolio:
        max   w'mu / sqrt(w'Sigma w)
        s.t.  sum(w) = 1,  0 <= w_i <= weight_cap
    Falls back to equal-weight if optimizer does not converge.
    """
    n   = len(mu)
    w0  = np.ones(n) / n
    cov_reg = cov + np.eye(n) * 1e-6

    def neg_sharpe(w):
        ret = w @ mu
        vol = np.sqrt(max(w @ cov_reg @ w, 1e-12))
        return -ret / vol

    result = minimize(
        neg_sharpe, w0,
        method='SLSQP',
        bounds=[(0.0, weight_cap)] * n,
        constraints=[{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}],
        options={'ftol': 1e-9, 'maxiter': 1000}
    )

    if result.success:
        w = np.clip(result.x, 0, weight_cap)
        return w / w.sum()
    return w0


def build_diagonal_cov(df, permnos, hist_months):
    """
    Diagonal covariance: each stock's own rolling return variance.
    Fallback = (20% annual vol / sqrt(12))^2 if history < 2 obs.
    """
    variances = []
    for p in permnos:
        h = df[(df['PERMNO'] == p) & (df['DateKey'].isin(hist_months))]['MthRet']
        variances.append(h.var() if len(h) >= 2 else (0.20 / np.sqrt(12)) ** 2)
    return np.diag(variances)


def rank_transform_mu(mu_raw, scale=0.10):
    """
    Replace raw predicted returns with cross-sectional rank scores
    mapped to [0, scale]. Preserves ordinal signal; the scalar is
    irrelevant to the MVO solution (multiplying all mu by a constant
    does not change the max-Sharpe weights).
    """
    ranks = mu_raw.argsort().argsort().astype(float)
    return (ranks / max(ranks.max(), 1)) * scale

## 3. Monthly MVO Loop

In [ ]:
def run_mvo(top6_df, label, weight_cap=WEIGHT_CAP, cov_window=COV_WINDOW):
    """
    For each month T:
      1. Rank-transform predicted returns → mu
      2. Estimate Sigma as diagonal matrix of rolling stock variances
      3. Solve max-Sharpe MVO → optimal weights w*
      4. Realized return  = w* . y_next
      5. Equal-weight return = (1/6) . y_next
    """
    all_months = sorted(top6_df['DateKey'].unique())
    ret_opt, ret_ew, datekeys = [], [], []

    for i, month in enumerate(all_months):
        md      = top6_df[top6_df['DateKey'] == month]
        permnos = md['PERMNO'].tolist()
        mu_raw  = md['predicted_return'].values
        y       = md['y_next'].values

        mu_sc = rank_transform_mu(mu_raw)

        start       = max(0, i - cov_window)
        hist_months = all_months[start:i]
        cov = build_diagonal_cov(top6_df, permnos, hist_months)

        # First month has no history: equal-weight fallback
        if i == 0:
            w = np.ones(len(permnos)) / len(permnos)
        else:
            w = max_sharpe_weights(mu_sc, cov, weight_cap)

        w_ew = np.ones(len(permnos)) / len(permnos)

        ret_opt.append(float(w    @ y))
        ret_ew.append( float(w_ew @ y))
        datekeys.append(month)

    print(f'[{label}] Done. {len(datekeys)} months | cap={weight_cap:.4f}')
    return datekeys, ret_opt, ret_ew


dk_ols, ols_opt, ols_ew = run_mvo(top6_ols, label='OLS')
dk_rf,  rf_opt,  rf_ew  = run_mvo(top6_rf,  label='RF')

## 4. Performance Summary

In [ ]:
def performance_metrics(ret_series, label=''):
    """Methodology §12.2 metrics from monthly excess return series."""
    r = np.array(ret_series)
    T = len(r)
    ann_ret = (1 + r).prod() ** (12 / T) - 1
    ann_vol = r.std() * np.sqrt(12)
    sharpe  = (r.mean() / r.std()) * np.sqrt(12)
    wealth  = (1 + r).cumprod()
    peak    = np.maximum.accumulate(wealth)
    mdd     = ((peak - wealth) / peak).max()
    return {'Portfolio'   : label,
            'Ann. Return' : f'{ann_ret:.2%}',
            'Ann. Vol'    : f'{ann_vol:.2%}',
            'Sharpe Ratio': f'{sharpe:.4f}',
            'Max Drawdown': f'{mdd:.2%}'}

metrics = [
    performance_metrics(ols_opt, 'OLS + MVO'),
    performance_metrics(rf_opt,  'RF  + MVO'),
    performance_metrics(ols_ew,  'Equal-Weight (OLS stocks)'),
    performance_metrics(rf_ew,   'Equal-Weight (RF stocks)'),
]

print('=== PORTFOLIO PERFORMANCE TABLE (Methodology §13.2) ===')
display(pd.DataFrame(metrics).set_index('Portfolio'))

## 5. Monthly Weight Inspection

In [ ]:
def show_weights(top6_df, datekeys, label, n_months=2):
    """
    Reconstruct and display per-stock weights for the first n_months.
    Verifies that MVO is producing differentiated (non-equal) weights
    and that weights are consistent with the rank-transformed signal.
    """
    all_months = sorted(top6_df['DateKey'].unique())
    sample = all_months[:n_months]
    rows = []
    for i, month in enumerate(all_months):
        if month not in sample:
            continue
        md      = top6_df[top6_df['DateKey'] == month]
        permnos = md['PERMNO'].tolist()
        mu_raw  = md['predicted_return'].values
        tickers = md['Ticker'].tolist()
        mu_sc   = rank_transform_mu(mu_raw)
        start       = max(0, i - COV_WINDOW)
        hist_months = all_months[start:i]
        cov = build_diagonal_cov(top6_df, permnos, hist_months)
        w = np.ones(len(permnos)) / len(permnos) if i == 0 \
            else max_sharpe_weights(mu_sc, cov, WEIGHT_CAP)
        for ticker, weight, signal in zip(tickers, w, mu_sc):
            rows.append({'DateKey': month, 'Ticker': ticker,
                         'Pred Signal': round(signal, 4),
                         'Weight': round(weight, 4)})
    df = pd.DataFrame(rows)
    print(f'[{label}] Weights — first {n_months} months:')
    display(
        df.sort_values(['DateKey', 'Weight'], ascending=[True, False])
          .reset_index(drop=True)
    )

show_weights(top6_ols, dk_ols, 'OLS+MVO')
show_weights(top6_rf,  dk_rf,  'RF+MVO')

## 6. Cumulative Wealth Curves

In [ ]:
import matplotlib.pyplot as plt

months = pd.to_datetime(
    pd.Series(dk_ols).astype(str), format='%Y%m'
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── OLS panel ────────────────────────────────────────────────────────────────
ax = axes[0]
ax.plot(months, (1 + pd.Series(ols_opt)).cumprod(),
        label='OLS + MVO (diag Σ, cap=1/3)', linewidth=2, color='steelblue')
ax.plot(months, (1 + pd.Series(ols_ew)).cumprod(),
        label='Equal-Weight', linewidth=2, linestyle='--', color='orange')
ax.axhline(1, color='grey', linewidth=0.8, linestyle=':')
ax.set_title('OLS Model: MVO vs Equal-Weight')
ax.set_ylabel('Cumulative Wealth ($1 invested)')
ax.legend(); ax.grid(True, alpha=0.3)

# ── RF panel ─────────────────────────────────────────────────────────────────
ax = axes[1]
ax.plot(months, (1 + pd.Series(rf_opt)).cumprod(),
        label='RF + MVO (diag Σ, cap=1/3)', linewidth=2, color='green')
ax.plot(months, (1 + pd.Series(rf_ew)).cumprod(),
        label='Equal-Weight', linewidth=2, linestyle='--', color='orange')
ax.axhline(1, color='grey', linewidth=0.8, linestyle=':')
ax.set_title('RF Model: MVO vs Equal-Weight')
ax.set_ylabel('Cumulative Wealth ($1 invested)')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('wealth_curves_v4.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. OLS vs RF Head-to-Head

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(months, (1 + pd.Series(ols_opt)).cumprod(),
        label='OLS + MVO', linewidth=2, color='steelblue')
ax.plot(months, (1 + pd.Series(rf_opt)).cumprod(),
        label='RF  + MVO', linewidth=2, color='green')
ax.plot(months, (1 + pd.Series(ols_ew)).cumprod(),
        label='Equal-Weight (OLS stocks)', linewidth=1.5,
        linestyle='--', alpha=0.7, color='orange')
ax.axhline(1, color='grey', linewidth=0.8, linestyle=':')
ax.set_title('Cumulative Wealth: OLS+MVO vs RF+MVO vs Equal-Weight (2022–)')
ax.set_ylabel('Cumulative Wealth')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('head_to_head_v4.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Save Monthly Returns

In [ ]:
monthly_ret = pd.DataFrame({
    'DateKey'      : dk_ols,
    'OLS_MVO'      : ols_opt,
    'RF_MVO'       : rf_opt,
    'EW_OLS_stocks': ols_ew,
    'EW_RF_stocks' : rf_ew,
})

monthly_ret.to_csv('mvo_monthly_returns_v4.csv', index=False)
display(monthly_ret)